# Bronze Layer - Olist E-Commerce Ingestion

Pulls the raw Olist CSVs from S3, lands them in a Unity Catalog Volume,
then loads each one into a bronze Delta table with an ingestion timestamp.

**Source:** S3 (`raw/Olist/`)
**Landing zone:** `/Volumes/workspace/default/raw_data/olist`
**Output:** Delta tables under `workspace.bronze`

## 1. Sync source files from S3

Volumes don't support the random-access writes boto3 uses for larger file
downloads, so files are staged in `/tmp` first, then copied over.

In [0]:
import shutil
import logging
from pathlib import Path

import boto3

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("bronze")

BUCKET_NAME = "ysa-ecommerce-lakehouse-2026-967228660879-eu-north-1-an"
SOURCE_PREFIX = "raw/Olist/"
STAGING_DIR = Path("/tmp/olist_staging")       # scratch space, gets wiped on cluster restart
VOLUME_DIR = Path("/Volumes/workspace/default/raw_data/olist")  # actual persistent landing zone

STAGING_DIR.mkdir(parents=True, exist_ok=True)
VOLUME_DIR.mkdir(parents=True, exist_ok=True)

# pulling creds from secrets, never hardcoding these
s3 = boto3.client(
    "s3",
    aws_access_key_id=dbutils.secrets.get(scope="aws", key="access_key"),
    aws_secret_access_key=dbutils.secrets.get(scope="aws", key="secret_key"),
    region_name="eu-north-1",
)

# list everything sitting under the Olist folder in the bucket
objects = s3.list_objects_v2(Bucket=BUCKET_NAME, Prefix=SOURCE_PREFIX).get("Contents", [])

for obj in objects:
    filename = obj["Key"].split("/")[-1]
    if not filename:
        continue  # S3 lists the folder itself as an empty-name object, skip it

    final_path = VOLUME_DIR / filename
    if final_path.exists():
        continue  # already synced this one on a previous run, don't redo it

    # download to /tmp first (boto3's multipart writes need normal disk),
    # then copy the finished file into the volume (a plain copy works fine there)
    staging_path = STAGING_DIR / filename
    s3.download_file(BUCKET_NAME, obj["Key"], str(staging_path))
    shutil.copy(staging_path, final_path)
    logger.info(f"synced {filename}")

## 2. Load raw files into bronze Delta tables

Each CSV becomes its own table under `workspace.bronze`, tagged with when it
was ingested. Using `overwrite` for now since this is a one-time batch load —
we'll switch to incremental loading (Auto Loader) once streaming comes in.

In [0]:
from pyspark.sql.functions import current_timestamp

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# maps each raw filename to the table name we want it stored as
tables = {
    "olist_customers_dataset.csv": "customers",
    "olist_geolocation_dataset.csv": "geolocation",
    "olist_order_items_dataset.csv": "order_items",
    "olist_order_payments_dataset.csv": "order_payments",
    "olist_order_reviews_dataset.csv": "order_reviews",
    "olist_orders_dataset.csv": "orders",
    "olist_products_dataset.csv": "products",
    "olist_sellers_dataset.csv": "sellers",
    "product_category_name_translation.csv": "category_translation",
}

for filename, table_name in tables.items():
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)   # fine for bronze, we'll enforce real types in silver
        .option("multiLine", True)     # review comments can span multiple lines
        .option("quote", '"')          # so embedded commas/newlines inside quotes don't split columns
        .option("escape", '"')
        .csv(str(VOLUME_DIR / filename))
        .withColumn("_ingested_at", current_timestamp())  # so we can trace when each load happened
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")  # schema can legitimately change now that parsing is fixed
        .saveAsTable(f"workspace.bronze.{table_name}")
    )
    print(f"loaded {table_name}: {df.count()} rows")

## 3. Sanity check

Confirm all 9 tables exist and spot-check one of them.

In [0]:
# should list 9 tables
display(spark.sql("SHOW TABLES IN workspace.bronze"))

In [0]:
# eyeball a few rows to make sure the data actually looks right
display(spark.sql("SELECT * FROM workspace.bronze.orders LIMIT 10"))